# Marathi-English Code-Mixed Hate Speech Detection
Fine-tuning a multilingual transformer (XLM-RoBERTa / MuRIL / mBERT) for binary classification (0 = None, 1 = Hate).

Put `marathi_english_code_mixed_sample.csv` in the same folder as this notebook (columns: `text,label,label_name`). Works on a laptop (CPU/Apple MPS) or Colab GPU.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib seaborn torch

In [ ]:
import os, re, random, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback)

# ---- Config ----
DATA_PATH  = "marathi_english_code_mixed_sample.csv"
# Options: "xlm-roberta-base" | "google/muril-base-cased" | "bert-base-multilingual-cased" | "ai4bharat/indic-bert"
MODEL_NAME = "xlm-roberta-base"
MAX_LEN, BATCH, EPOCHS, LR, SEED = 64, 8, 10, 2e-5, 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("Device:", "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

## 1. Load & explore data

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df.dropna(subset=["text","label"]).drop_duplicates(subset="text").reset_index(drop=True)
df["label"] = df["label"].astype(int)
print(df.shape); display(df.head())
sns.countplot(x="label_name", data=df); plt.title("Class distribution"); plt.show()

## 2. Preprocess & split (stratified 70/15/15)

In [ ]:
def clean(t):
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))   # urls
    t = re.sub(r"@\w+", "@user", t)                 # mentions
    t = re.sub(r"\s+", " ", t).strip()
    return t
df["text"] = df["text"].apply(clean)

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df["label"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

## 3. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def to_ds(d):
    ds = Dataset.from_pandas(d[["text","label"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
train_ds, val_ds, test_ds = map(to_ds, (train_df, val_df, test_df))

## 4. Model & training

In [ ]:
id2label = {0:"None", 1:"Hate"}; label2id = {v:k for k,v in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

import math
WARMUP_STEPS = int(0.1 * math.ceil(len(train_ds)/BATCH) * EPOCHS)  # ~10% warmup; works on all transformers versions

args = TrainingArguments(
    output_dir="./results", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    weight_decay=0.01, warmup_steps=WARMUP_STEPS,
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", save_total_limit=2,
    seed=SEED, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer.train()

## 5. Evaluate on test set

In [ ]:
out = trainer.predict(test_ds)
y_pred, y_true = np.argmax(out.predictions, axis=1), out.label_ids
print(classification_report(y_true, y_pred, labels=[0,1], target_names=["None","Hate"], zero_division=0))
sns.heatmap(confusion_matrix(y_true, y_pred, labels=[0,1]), annot=True, fmt="d", cmap="Blues",
            xticklabels=["None","Hate"], yticklabels=["None","Hate"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix"); plt.show()

## 6. Save model & predict on new text

In [ ]:
trainer.save_model("hate_speech_model"); tokenizer.save_pretrained("hate_speech_model")

def predict(texts):
    if isinstance(texts, str): texts = [texts]
    model.eval()
    enc = tokenizer([clean(t) for t in texts], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
    return [{"text":t, "label":id2label[int(p.argmax())], "confidence":float(p.max())} for t,p in zip(texts, probs)]

predict(["Tu lokanna insult karna band kar", "Aaj weather mast aahe", "That community is disgusting"])

**Notes**
- The sample has only 30 rows, so scores will be noisy. Use your full dataset for real results: just change `DATA_PATH`.
- For better Marathi/Hinglish performance try `MODEL_NAME = "google/muril-base-cased"`.
- With imbalanced data, add class weights or report macro-F1 (already used for model selection).